# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

volatility_flag_success completed
vix_full_sample_lookahead completed
arima_forecast_block blocked


prompt_injection_block blocked
spread_frequency_mismatch_block blocked


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

37 run records, 0 counted by the page, 401 events


total              37.000000
completed          17.000000
blocked            17.000000
failed              0.000000
rejected            0.000000
completed_share     0.459459
blocked_share       0.459459
failed_share        0.000000
rejected_share      0.000000
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

In [4]:
from quantlab.monitoring import (
    grounded_narratives,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **planner_refusals(records),
    }
)

approved             14.000000
rejected              0.000000
approval_rate         1.000000
edit_rate             0.000000
approval_median_s     0.040500
approval_p95_s        0.047450
replays               3.000000
replay_match_rate     1.000000
narratives           17.000000
grounded_rate         1.000000
refused              12.000000
refused_share         0.324324
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,42
1,P01,PLANNER_REFUSED,BLOCK,12
2,V09,LOOK_AHEAD,APPROVE,7
3,V08,FREQUENCY_MISMATCH,BLOCK,5
4,V03,INVALID_PARAM,BLOCK,2
5,E04,ALIGNMENT_LOSS,WARN,1
6,R01,REQUEST_TOO_LONG,BLOCK,0
7,V01,INVALID_PLAN,BLOCK,0
8,V02,UNKNOWN_CAPABILITY,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,18
1,flag_regime,C07,16
2,resample,C02,12
3,compute_returns,C03,10
4,rolling_volatility,C06,10
5,forward_return,C08,10
6,compare_groups,C09,10
7,hac_mean_difference,C10,10
8,spread,C05,1
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,6,3652.5,4435.5,280.500000,792.50,0.000505,0.000836,0.003033
1,narrator,6,766.5,1578.0,81.166667,143.50,0.000117,0.000230,0.000703
2,total,6,4419.0,5788.5,361.666667,922.75,0.000623,0.001040,0.003736


planner_calls             7.000000
cache_hits                1.000000
hit_rate                  0.142857
saved_input_tokens     4380.000000
saved_output_tokens     931.000000
saved_cost                0.000904
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,37,825.029730,5461.280,True
2,Validator,37,0.130541,0.402,True
3,Approval,14,39.500000,47.450,True
4,Executor,17,115.300000,164.326,True
5,Narrator,17,725.911765,3017.900,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-18T16:47:22.625Z,20261018-164722-4ca396,blocked,Validator,V08,The plan has blocking issues: V08:s4
1,2026-10-18T16:47:22.598Z,20261018-164722-38d62e,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-18T16:47:22.570Z,20261018-164722-3c3ba9,blocked,Planner,P01,The plan has blocking issues: P01:plan
3,2026-10-18T16:05:44.748Z,20261018-160544-7416f9,blocked,Validator,V08,The plan has blocking issues: V08:s4
4,2026-10-18T16:05:44.713Z,20261018-160544-4473b0,blocked,Planner,P01,The plan has blocking issues: P01:plan
